# Chunk Review Notebook

这个 notebook 用于人工审查 chunk 切分质量。

打开后即可直接运行，默认内置 3 段英文样本，分别覆盖：

- 叙述型长文
- 列表/报告型结构
- 混合格式与伪表格结构

执行顺序固定为：先看 `ChunkSplitter.split_paragraphs()`，再看 `ChunkSelector.select_chunks()`。

另外支持手动粘贴一段文本内容，作为额外审查样本。

In [1]:
from __future__ import annotations

import hashlib
import re
import sys
from collections import OrderedDict
from pathlib import Path
from typing import Iterable
import os

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'embedding').exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from embedding.services.chunking.chunk_selector import ChunkSelector
from embedding.services.chunking.chunk_splitter import ChunkSplitter
from embedding.services.chunking.selector_config import SelectorConfig

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

pd.set_option('display.max_colwidth', 140)
pd.set_option('display.width', 180)


In [2]:
samples = OrderedDict([
    (
        'narrative',
        '''
Deployment review notes from the weekly operations meeting. The team started by summarizing the rollout plan for the new embedding pipeline, then reviewed the expected runtime impact, cache behavior, and the order in which datasets would be processed. Everyone agreed that the first execution could be slower because models might need to be downloaded before chunking begins.

The discussion then shifted to text splitting quality. One engineer described a long document with multiple paragraphs, where each paragraph carried a different operational concern. Another engineer raised a separate point about long sentences that contain several clauses, especially when those clauses are separated by commas rather than line breaks. The group wanted to make sure such text would stay readable after splitting instead of being fragmented into tiny pieces.

By the end of the meeting, the conclusion was straightforward: preserve paragraph boundaries whenever they are meaningful, avoid breaking stable sections into smaller fragments without reason, and only split aggressively when the content truly becomes too long to process safely. That guideline was written down as the default review rule for the sample set.
'''.strip(),
    ),
    (
        'report_with_lists',
        '''
Operations Review Summary

Purpose:
This sample checks whether the splitter keeps report-style text readable when it contains headings, bullets, numbered steps, and short closing remarks.

Observed findings:
- A heading followed by a short explanatory sentence should remain grouped as a coherent block.
- Bullet items should stay attached to the section that introduces them.
- A short concluding paragraph should not be isolated from the note that precedes it.

Action items:
1. Confirm that paragraph breaks are respected.
2. Verify that list items are not split into unrelated fragments.
3. Check whether short lines at the end of the section remain readable after splitting.

Additional notes:
The review text intentionally mixes short and long sentences.
Some lines are compact and direct.
Other lines are deliberately longer, with several clauses and extra detail, so that the splitting logic has to decide whether to keep them intact or break them only at a sensible boundary.
'''.strip(),
    ),
    (
        'mixed_format',
        '''
Incident Review Log

Document: Embedding pipeline sample
Category: mixed-format operational note
Expected behavior: preserve logical blocks where possible

Summary:
This document mixes prose, a pseudo-table, and a closing recommendation. The goal is to see whether chunk splitting keeps each block stable enough for review while still separating sections that should not be merged together.

Pseudo-table:
Field | Value | Comment
Model | bge-m3 | Local encoder selected for review
Dataset | HotpotQA | Representative benchmark input
Mode | evaluation | Expected to reuse the shared cache

Detailed observations:
The first paragraph is intentionally short and acts as a section opener. The second paragraph is longer and contains several clauses that describe cache initialization, dataset processing, and chunk selection behavior. The third paragraph follows after a blank line so that the splitting logic has to respect the visible separation rather than blending everything together.

Final recommendation:
If the splitter behaves well, the pseudo-table should remain easy to read, the longer paragraph should not be cut too early, and the closing remark should still feel like a distinct summary instead of being merged into unrelated content.
'''.strip(),
    ),
])

# 手动粘贴一段文本即可；留空则不会额外添加样本。
manual_text = """
""".strip()


In [3]:
def preview_text(text: str, limit: int = 120) -> str:
    return text[:limit].replace('\n', '\\n')


def render_split_table(splitter: ChunkSplitter, text: str, sample_name: str) -> pd.DataFrame:
    paragraphs = splitter.split_paragraphs(text)
    rows = []
    for idx, paragraph in enumerate(paragraphs):
        rows.append({
            'sample': sample_name,
            'idx': idx,
            'char_len': len(paragraph),
            'preview': preview_text(paragraph),
            'text': paragraph,
        })
    return pd.DataFrame(rows)


def render_select_table(selector: ChunkSelector, text: str, sample_name: str) -> pd.DataFrame:
    selected = selector.select_chunks(text, title=sample_name)
    rows = []
    for idx, item in enumerate(selected):
        embedding = item.get('embedding', [])
        rows.append({
            'sample': sample_name,
            'idx': idx,
            'score': float(item.get('score', 0.0)),
            'embedding_dim': len(embedding),
            'preview': preview_text(item['chunk_text']),
            'chunk_text': item['chunk_text'],
        })
    return pd.DataFrame(rows)


class NotebookEmbeddingStrategy:
    def __init__(self, dim: int = 16):
        self.dim = dim

    def encode(self, texts: Iterable[str], is_query: bool):
        vectors = []
        for text in texts:
            text = text or ''
            tokens = re.findall(r'[A-Za-z0-9]+', text.lower())
            vec = np.zeros(self.dim, dtype=np.float32)
            for token in tokens:
                digest = hashlib.sha1(token.encode('utf-8')).hexdigest()
                vec[int(digest[:8], 16) % self.dim] += 1.0
            vec[0] += len(text)
            vec[1] += len(tokens)
            vec[2] += text.count('\n')
            vec[3] += text.count('.') + text.count('!') + text.count('?')
            if is_query:
                vec[4] += 1.0
            vectors.append(vec)
        return np.asarray(vectors, dtype=np.float32)


all_samples = OrderedDict(samples)
if manual_text.strip():
    all_samples['manual_text'] = manual_text.strip()

splitter = ChunkSplitter(min_sentences=3, max_tokens=8092)
selector = ChunkSelector(
    embedding_strategy=NotebookEmbeddingStrategy(),
    chunk_num=3,
    config=SelectorConfig(),
)

display(Markdown(f'## Loaded samples: {len(all_samples)}'))
list(all_samples.keys())


## Loaded samples: 3

['narrative', 'report_with_lists', 'mixed_format']

In [4]:
for sample_name, text in all_samples.items():
    display(Markdown(f'### Sample: `{sample_name}`'))
    display(Markdown('**Original text**'))
    display(Markdown(text.replace('\n', '  \n')))

    display(Markdown('**ChunkSplitter.split_paragraphs()**'))
    display(render_split_table(splitter, text, sample_name))

    display(Markdown('**ChunkSelector.select_chunks()**'))
    display(render_select_table(selector, text, sample_name))

    display(Markdown('---'))


### Sample: `narrative`

**Original text**

Deployment review notes from the weekly operations meeting. The team started by summarizing the rollout plan for the new embedding pipeline, then reviewed the expected runtime impact, cache behavior, and the order in which datasets would be processed. Everyone agreed that the first execution could be slower because models might need to be downloaded before chunking begins.  
  
The discussion then shifted to text splitting quality. One engineer described a long document with multiple paragraphs, where each paragraph carried a different operational concern. Another engineer raised a separate point about long sentences that contain several clauses, especially when those clauses are separated by commas rather than line breaks. The group wanted to make sure such text would stay readable after splitting instead of being fragmented into tiny pieces.  
  
By the end of the meeting, the conclusion was straightforward: preserve paragraph boundaries whenever they are meaningful, avoid breaking stable sections into smaller fragments without reason, and only split aggressively when the content truly becomes too long to process safely. That guideline was written down as the default review rule for the sample set.

**ChunkSplitter.split_paragraphs()**

,sample,idx,char_len,preview,text
0,narrative,0,375,Deployment review notes from the weekly operations meeting. The team started by summarizing the rollout plan for the new,Deployment review notes from the weekly operations meeting. The team started by summarizing the rollout plan for the new embedding pipel...
1,narrative,1,474,"The discussion then shifted to text splitting quality. One engineer described a long document with multiple paragraphs,","The discussion then shifted to text splitting quality. One engineer described a long document with multiple paragraphs, where each parag..."
2,narrative,2,358,"By the end of the meeting, the conclusion was straightforward: preserve paragraph boundaries whenever they are meaningfu","By the end of the meeting, the conclusion was straightforward: preserve paragraph boundaries whenever they are meaningful, avoid breakin..."


...

,sample,idx,char_len,preview,text
0,mixed_format,0,153,Incident Review Log\nDocument: Embedding pipeline sample\nCategory: mixed-format operational note\nExpected behavior: prese,Incident Review Log\nDocument: Embedding pipeline sample\nCategory: mixed-format operational note\nExpected behavior: preserve logical b...
1,mixed_format,1,430,"Summary:\nThis document mixes prose, a pseudo-table, and a closing recommendation. The goal is to see whether chunk split","Summary:\nThis document mixes prose, a pseudo-table, and a closing recommendation. The goal is to see whether chunk splitting keeps each..."
2,mixed_format,2,396,Detailed observations:\nThe first paragraph is intentionally short and acts as a section opener. The second paragraph is,Detailed observations:\nThe first paragraph is intentionally short and acts as a section opener. The second paragraph is longer and cont...
3,mixed_format,3,259,"Final recommendation:\nIf the splitter behaves well, the pseudo-table should remain easy to read, the longer paragraph sh","Final recommendation:\nIf the splitter behaves well, the pseudo-table should remain easy to read, the longer paragraph should not be cut..."


**ChunkSelector.select_chunks()**

/Users/zhangjie/anaconda3/envs/stock_analysis/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['d', 'll', 'm', 'n', 's', 't', 've', '且', '东', '个', '为', '么', '事', '些', '什', '以', '们', '但', '候', '哪', '因', '地', '怎', '情', '或', '所', '方', '时', '样', '者', '而', '西', '还', '这', '那', '里', '问', '题'] not in stop_words.
  warnings.warn(


,sample,idx,score,embedding_dim,preview,chunk_text
0,mixed_format,0,0.600444,16,"Summary:\nThis document mixes prose, a pseudo-table, and a closing recommendation. The goal is to see whether chunk split","Summary:\nThis document mixes prose, a pseudo-table, and a closing recommendation. The goal is to see whether chunk splitting keeps each..."
1,mixed_format,1,0.354377,16,"Final recommendation:\nIf the splitter behaves well, the pseudo-table should remain easy to read, the longer paragraph sh","Final recommendation:\nIf the splitter behaves well, the pseudo-table should remain easy to read, the longer paragraph should not be cut..."
2,mixed_format,2,0.224918,16,Detailed observations:\nThe first paragraph is intentionally short and acts as a section opener. The second paragraph is,Detailed observations:\nThe first paragraph is intentionally short and acts as a section opener. The second paragraph is longer and cont...


---

手动粘贴文本时，只需要修改上面的 `manual_text` 单元。

如果你要看真实模型下的结果，可以把 `NotebookEmbeddingStrategy()` 替换成项目里的实际 embedding strategy。